# VAEモデルにおける正則化技術

このノートブックでは、Variational Autoencoder (VAE) モデルに使用されている正則化技術について説明します。正則化は過学習を防ぎ、モデルの汎化性能を向上させるための重要な手法です。

## バッチ正規化 (Batch Normalization)

バッチ正規化は、ニューラルネットワークの各層の出力を正規化する技術です。この手法は2015年にIoffe and Szegedy によって提案され、深層ニューラルネットワークのトレーニングを安定化させ、高速化する効果があります。

### バッチ正規化の仕組み

1. ミニバッチ内の各特徴（ニューロン出力）について、平均μと標準偏差σを計算
2. 各特徴値を正規化：$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$
3. スケールとシフトパラメータを適用：$y = \gamma \hat{x} + \beta$

ここで、γとβは学習可能なパラメータで、ネットワークがトレーニング中に最適な値を学習します。

### バッチ正規化の利点

- **勾配消失・爆発の防止**: 深層ネットワークでの勾配フローを改善
- **学習の高速化**: より大きな学習率が使用可能に
- **重みの初期化への依存度低減**: 初期値に対する敏感さの軽減
- **正則化効果**: ミニバッチ内のデータのランダム性が正則化効果をもたらす

### このVAEモデルでのバッチ正規化

このモデルでは、エンコーダとデコーダの各層に`nn.BatchNorm1d`が適用されています：

```python
self.encoder = nn.Sequential(
    nn.Linear(input_dim, hidden_dim),
    nn.BatchNorm1d(hidden_dim),  # 第1層のバッチ正規化
    nn.ReLU(),
    nn.Dropout(0.3),
    # ...以下同様
)
```

バッチ正規化は活性化関数（ReLU）の前に適用されることもありますが、このモデルでは線形層の直後、ReLUの前に配置されています。

## ドロップアウト (Dropout)

ドロップアウトは、トレーニング中にランダムにニューロンを「脱落」させる正則化技術です。この手法は2012年にHinton et al.によって提案されました。

### ドロップアウトの仕組み

1. トレーニング時、各ニューロンは指定された確率pで一時的に無効化（出力を0に）
2. 残りの(1-p)の確率でニューロンは通常通り動作するが、出力は1/(1-p)倍される
3. 推論時（テスト時）はドロップアウトを適用せず、すべてのニューロンを使用

### ドロップアウトの利点

- **共適応の防止**: ニューロン間の過度な依存関係を減少
- **アンサンブル効果**: 複数のサブネットワークをトレーニングして平均化する効果
- **過学習の抑制**: モデルの表現力を制限し、より汎用的な特徴の学習を促進

### このVAEモデルでのドロップアウト

このモデルでは、エンコーダとデコーダの複数の層に`nn.Dropout`が適用されています：

```python
self.encoder = nn.Sequential(
    # ...
    nn.ReLU(),
    nn.Dropout(0.3),  # 30%のニューロンを無効化
    # ...
)
```

エンコーダでは、第1層と第2層の後に30%のドロップアウト率、第3層の後に20%のドロップアウト率が設定されています。デコーダも同様に各層にドロップアウトが適用されています。

## 勾配クリッピング (Gradient Clipping)

勾配クリッピングは、勾配爆発問題（gradient explosion）を防ぐために勾配のノルム（大きさ）に上限を設ける技術です。

### 勾配クリッピングの仕組み

1. ミニバッチごとの勾配更新の前に、すべてのパラメータの勾配ノルムを計算
2. 勾配ノルムが閾値を超える場合、勾配全体を再スケーリングして閾値以下に調整

### 勾配クリッピングの利点

- **トレーニングの安定化**: 急激な勾配の変化を抑え、学習の発散を防止
- **勾配爆発の防止**: 特に再帰的ニューラルネットワーク（RNN）で重要
- **極端な更新の抑制**: 極端に大きな勾配によるパラメータの破壊的更新を防ぐ

### このVAEモデルでの勾配クリッピング

トレーニングループ内で各バッチ処理後に勾配クリッピングが適用されています：

```python
loss.backward()
# 勾配クリッピング（安定化のため）
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

ここでは、すべてのパラメータの勾配ノルムが1.0を超えないようにクリッピングされています。

## β-VAE (Beta-VAE)

β-VAEは標準的なVAEの拡張で、潜在空間の分離性（disentanglement）を改善するためにKLダイバージェンス項に重み付けを行います。

### β-VAEの仕組み

標準的なVAEの損失関数は以下の2項から構成されます：
1. 再構成誤差：入力と再構成出力の差
2. KLダイバージェンス：潜在変数の分布と標準正規分布の差

β-VAEでは、この2つの項のバランスをβパラメータで制御します：

$$\mathcal{L} = \mathcal{L}_{\text{recon}} + \beta \cdot \mathcal{L}_{\text{KL}}$$

βが1より大きい場合、KL項が強調され、より分離された潜在表現が得られます。
βが1より小さい場合、再構成誤差が優先され、より正確な再構成が得られます。

### β-VAEの利点

- **潜在変数の分離性制御**: βの値によって特徴の分離度を調整可能
- **表現学習の柔軟性**: タスクに応じて再構成精度と分離性のトレードオフを調整可能
- **解釈可能な潜在空間**: より分離された潜在変数は個別の意味を持ちやすい

### このVAEモデルでのβ-VAE

このモデルでは、βの値が0.5に設定されています：

```python
beta = 0.5  # β-VAEパラメータ（KL項の重みを小さく）
```

そして、損失関数内でこのβ値が使用されています：

```python
def wavelet_vae_loss(recon_x, x, mu, logvar, beta=1.0):
    # 再構成誤差（MSE）
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    
    # KLダイバージェンス
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + beta * kld_loss, recon_loss, kld_loss
```

βが0.5に設定されていることから、このモデルは再構成精度を優先しており、KLダイバージェンス項の影響を抑えています。

## 重み減衰 (Weight Decay)

重み減衰は、ネットワークの重みパラメータに対してL2正則化を適用することで、過学習を防ぐ技術です。

### 重み減衰の仕組み

1. 損失関数に重みパラメータの二乗和の項を追加：$\mathcal{L}_{\text{reg}} = \mathcal{L} + \lambda \sum_i w_i^2$
2. これにより、重みが大きくなりすぎることを防ぎ、より単純なモデルが優先される

### 重み減衰の利点

- **過学習の抑制**: 複雑すぎるモデルの学習を防止
- **重みの制約**: パラメータが極端に大きくなることを防ぐ
- **一般化性能の向上**: より滑らかな決定境界を持つモデルを促進

### このVAEモデルでの重み減衰

オプティマイザの初期化時に重み減衰パラメータが設定されています：

```python
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
```

ここでは、重み減衰の強さを1e-5（0.00001）に設定しています。これは比較的小さな値であり、重み減衰の効果を穏やかに保ちながら過学習を防止します。

## まとめ

このVAEモデルでは、以下の正則化技術が組み合わせて使用されています：

1. **バッチ正規化**: 各層の出力を正規化し、学習を安定化・高速化
2. **ドロップアウト**: ランダムにニューロンを無効化し、過学習を防止
3. **勾配クリッピング**: 勾配の大きさを制限し、トレーニングを安定化
4. **β-VAE**: KLダイバージェンス項の重みを調整（β=0.5）して再構成精度を優先
5. **重み減衰**: 重みパラメータにL2正則化を適用（1e-5）

これらの技術を組み合わせることで、高次元のウェーブレット特徴量を効果的に低次元の潜在空間に圧縮し、モデルの過学習を防ぎながら良好な再構成性能を実現しています。